In [ ]:
from time import time

import matplotlib.pyplot as plt
import numpy as np
from hetero_isas.monodromy_lp import (
    MonodromyLPDecomposer,
)
from hetero_isas.monodromy_lp.decomposer import MonodromyLPDecomposerResult
from hetero_isas.monodromy_lp.invariants import recover_local_equivalence
from hetero_isas.monodromy_lp.isa import ISAHandler
from hetero_isas.monodromy_lp.mono_lp_result import plot_histograms
from hetero_isas.zz_parallel_drive.ansatz import BasicCircuitAnsatz
from numpy.random import Philox
from qiskit import QuantumCircuit
from qiskit.circuit import Gate, Parameter
from qiskit.circuit.library import CXGate, CZGate, SwapGate, UnitaryGate, iSwapGate
from qiskit.quantum_info import Operator, average_gate_fidelity, random_unitary
from qiskit.synthesis.two_qubit.local_invariance import two_qubit_local_invariants
from qutip import Qobj
from tqdm.notebook import tqdm
from weylchamber import c1c2c3
from hetero_isas.monodromy_lp.gate import MonodromyLPGate

generator = Philox(0)

%matplotlib inline

In [ ]:
## hacky way to determine if a LP call is feasible
# and then time it

X = 10_000

scipy_decomposer = MonodromyLPDecomposer(isa_handler, include_rho_target=False)
feasible_times = []
infeasible_times = []
for _ in tqdm(range(X)):
    target_unitary = random_unitary(4).to_matrix()
    target_gate = MonodromyLPGate.from_unitary(target_unitary, alcove_norm=False)
    rho_target = target_gate.rho_reflect()

    feasible_on_t = True
    feasible_on_rho = True

    # first, determine feasiblity criterion
    try:
        scipy_decomposer._best_decomposition(target_gate)
    except:
        feasible_on_t = False

    try:
        scipy_decomposer._best_decomposition(rho_target)
    except:
        feasible_on_rho = False

    # second time the feasible/infeasible LP calls
    if feasible_on_rho:
        start_time = time()
        scipy_decomposer._best_decomposition(rho_target)
        feasible_times.append(time() - start_time)

    else:
        try:
            start_time = time()
            scipy_decomposer._best_decomposition(rho_target)
        except:
            pass
        infeasible_times.append(time() - start_time)

    if feasible_on_t:
        start_time = time()
        scipy_decomposer._best_decomposition(target_gate)
        feasible_times.append(time() - start_time)

    else:
        try:
            start_time = time()
            scipy_decomposer._best_decomposition(target_gate)
        except:
            pass
        infeasible_times.append(time() - start_time)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
bins = 16
ax.hist(feasible_times, bins, label="feasible", color="green", alpha=0.8)
mean = np.mean(feasible_times)
ax.axvline(mean, color="green", linestyle="-", linewidth=1, alpha=0.7)
ax.hist(infeasible_times, bins, label="infeasible", color="red", alpha=0.8)
mean = np.mean(infeasible_times)
ax.axvline(mean, color="red", linestyle="-", linewidth=1, alpha=0.7)

fig.legend()

In [ ]:
M = 36
isa_handler = ISAHandler(
    [
        CXGate().power(1 / 24),
    ]
    * M,
    [1 / 24] * M,
    ["sq[24]cx"] * M,
    compute_coverage_set=False,
)

isa_handler.isa._fixed_sequence_length = M

decomposer = MonodromyLPDecomposer(isa_handler, True, False, False, False)

feasible_times = []
infeasible_times = []
for _ in tqdm(range(X)):
    target_unitary = random_unitary(4).to_matrix()
    target_gate = MonodromyLPGate.from_unitary(target_unitary, alcove_norm=False)
    feasible_on_t = True
    # first, determine feasiblity criterion
    try:
        decomposer._best_decomposition(target_gate)
    except:
        feasible_on_t = False

    if feasible_on_t:
        start_time = time()
        decomposer._best_decomposition(target_gate)
        feasible_times.append(time() - start_time)

    else:
        try:
            start_time = time()
            decomposer._best_decomposition(target_gate)
        except:
            pass
        infeasible_times.append(time() - start_time)

In [ ]:
# duration historams
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
bins = 64
amax = 0.0125  # 0.002  # 0.08

colors = ["green", "red"]

# histogram data and colors
data = [
    (feasible_times, "feasible"),
    (infeasible_times, "infeasible"),
]

for (durations, label), color in zip(data, colors):
    clipped_data = np.clip(durations, a_min=0, a_max=amax)
    ax.hist(clipped_data, bins, label=label, alpha=0.8, color=color)
    mean = np.mean(durations)
    std = np.std(durations)

    ax.axvline(mean, color=color, linestyle="-", linewidth=1, alpha=0.7)
    ax.axvline(mean + std / 2.0, color=color, linestyle="--", linewidth=1, alpha=0.1)
    ax.axvline(mean - std / 2.0, color=color, linestyle="--", linewidth=1, alpha=0.1)

ax.set_ylabel("Frequency")
ax.set_xlabel("Duration (s)")
ax.legend()